# DeBERTa-v3-large

Mi mejor modelo individual entre los encoder transformers. Lo entreno en cinco folds usando el texto enriquecido y dejo guardadas las probabilidades OOF y test para combinarlas más adelante con Mistral en el voting.

In [ ]:
import os, gc, numpy as np, pandas as pd, torch, torch.nn as nn
from transformers import AutoTokenizer, AutoModel, get_cosine_schedule_with_warmup
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score

MODEL_NAME = 'microsoft/deberta-v3-large'
MAX_LEN = 160
BATCH   = 8
GRAD_ACC = 2
EPOCHS  = 3
LR      = 1e-5
SEED    = 42

TRAIN_PATH = '/kaggle/input/2025-26-false-political-claim-detection/train.csv'
TEST_PATH  = '/kaggle/input/2025-26-false-political-claim-detection/test_nolabel.csv'

train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)
y = train['label'].values

Elegí DeBERTa-v3-large en lugar de DeBERTa-v3-base porque al probar base obtuve métricas notablemente peores y, aunque large tarda casi 2 horas en T4, el salto de F1 lo compensa. MAX_LEN 160 lo fijé después de mirar el percentil 95 de longitud, con 160 tokens cubro el statement entero más toda la cabecera de metadatos.

In [ ]:
def build_text(row):
    parts = []
    for col, prefix in [('speaker','Speaker'), ('party_affiliation','Party'),
                        ('speaker_job','Job'), ('state_info','State'),
                        ('subject','Topic')]:
        v = row.get(col)
        if pd.notna(v) and str(v).strip() != '':
            parts.append(f'{prefix}: {v}')
    head = ' | '.join(parts)
    return f"{head} [SEP] Claim: {row['statement']}"

train['text_enriched'] = train.apply(build_text, axis=1)
test['text_enriched']  = test.apply(build_text, axis=1)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
folds = np.zeros(len(train), dtype=int)
for f, (_, vi) in enumerate(skf.split(train, y)):
    folds[vi] = f

Replico la misma función build_text y los mismos folds del notebook 00 para garantizar que las predicciones OOF sean combinables después con las de los otros modelos.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class FakeClaimDs(Dataset):
    def __init__(self, texts, labels=None):
        self.texts = texts
        self.labels = labels
    def __len__(self): return len(self.texts)
    def __getitem__(self, i):
        enc = tokenizer(self.texts[i], truncation=True, max_length=MAX_LEN,
                        padding='max_length', return_tensors='pt')
        item = {k: v.squeeze(0) for k,v in enc.items()}
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[i], dtype=torch.long)
        return item

In [ ]:
class DebertaCls(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(MODEL_NAME)
        self.backbone.gradient_checkpointing_enable()
        self.dropout = nn.Dropout(0.1)
        self.head = nn.Linear(self.backbone.config.hidden_size, 2)
    def forward(self, input_ids, attention_mask):
        h = self.backbone(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state[:,0]
        return self.head(self.dropout(h))

La cabeza es simple: dropout + lineal a 2 clases sobre el embedding [CLS]. Probé añadir una capa intermedia y no mejoró, así que me quedé con la versión más simple. Activo gradient checkpointing porque sin él no entra en T4 16GB; me cuesta alrededor de un 30 % más de tiempo de entrenamiento, pero es asumible.

In [ ]:
def train_fold(fold):
    tr_idx = np.where(folds != fold)[0]
    va_idx = np.where(folds == fold)[0]
    ds_tr = FakeClaimDs(train['text_enriched'].values[tr_idx], y[tr_idx])
    ds_va = FakeClaimDs(train['text_enriched'].values[va_idx], y[va_idx])
    dl_tr = DataLoader(ds_tr, batch_size=BATCH, shuffle=True, num_workers=2)
    dl_va = DataLoader(ds_va, batch_size=BATCH*2, shuffle=False, num_workers=2)

    model = DebertaCls().cuda()
    optim = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    total = len(dl_tr) * EPOCHS // GRAD_ACC
    sched = get_cosine_schedule_with_warmup(optim, int(0.1*total), total)
    class_w = torch.tensor([1.85, 1.0]).cuda()
    loss_fn = nn.CrossEntropyLoss(weight=class_w, label_smoothing=0.05)

    best_f1, best_oof = 0, None
    for ep in range(EPOCHS):
        model.train()
        for step, batch in enumerate(dl_tr):
            ids = batch['input_ids'].cuda()
            am  = batch['attention_mask'].cuda()
            lab = batch['labels'].cuda()
            logits = model(ids, am)
            loss = loss_fn(logits, lab) / GRAD_ACC
            loss.backward()
            if (step+1) % GRAD_ACC == 0:
                optim.step(); sched.step(); optim.zero_grad()
        model.eval()
        probs = []
        with torch.no_grad():
            for batch in dl_va:
                logits = model(batch['input_ids'].cuda(), batch['attention_mask'].cuda())
                probs.append(torch.softmax(logits, -1)[:,1].cpu().numpy())
        probs = np.concatenate(probs)
        f1 = f1_score(y[va_idx], (probs>0.5).astype(int), average='macro')
        print(f'fold {fold} ep {ep}  F1={f1:.4f}')
        if f1 > best_f1:
            best_f1, best_oof = f1, probs
            torch.save(model.state_dict(), f'deberta_large_fold{fold}.bin')
    return best_oof

Elegí lr=1e-5 porque con 2e-5 (que sí funcionaba en base) los pesos de large se desestabilizaban en la primera época. Warmup ratio 0.1 por la misma razón: large necesita más calentamiento antes de subir el LR del todo. Class weights (1.85, 1.0) salen del inverso de las frecuencias 35/65, y label_smoothing 0.05 lo añadí para que las probabilidades estén mejor calibradas — me importa, porque luego se reutilizan en el voting.

In [ ]:
oof = np.zeros(len(train))
for f in range(5):
    np.random.seed(SEED+f); torch.manual_seed(SEED+f)
    oof_f = train_fold(f)
    oof[folds==f] = oof_f
    gc.collect(); torch.cuda.empty_cache()

best_t, best_f1 = 0.5, 0
for t in np.arange(0.30, 0.70, 0.01):
    s = f1_score(y, (oof>t).astype(int), average='macro')
    if s > best_f1: best_f1, best_t = s, t
print(f'OOF F1 = {best_f1:.4f}  thr {best_t:.2f}')

In [ ]:
test_pred = np.zeros(len(test))
ds_te = FakeClaimDs(test['text_enriched'].values)
dl_te = DataLoader(ds_te, batch_size=BATCH*2, shuffle=False, num_workers=2)
for f in range(5):
    model = DebertaCls().cuda()
    model.load_state_dict(torch.load(f'deberta_large_fold{f}.bin'))
    model.eval()
    fold_probs = []
    with torch.no_grad():
        for batch in dl_te:
            logits = model(batch['input_ids'].cuda(), batch['attention_mask'].cuda())
            fold_probs.append(torch.softmax(logits,-1)[:,1].cpu().numpy())
    test_pred += np.concatenate(fold_probs) / 5
    del model; gc.collect(); torch.cuda.empty_cache()

Para el test promedio las predicciones de los 5 modelos (uno por fold). Es bagging implícito y me reduce varianza sin coste adicional, porque los modelos ya están entrenados.

In [ ]:
np.save('oof_deberta_large.npy', oof)
np.save('test_deberta_large.npy', test_pred)

sub = pd.read_csv('/kaggle/input/2025-26-false-political-claim-detection/sample_submission.csv')
sub['label'] = (test_pred > best_t).astype(int)
sub.to_csv('submission_deberta_large.csv', index=False)
sub.head()

Guardo OOF y test en .npy para combinarlas en el siguiente notebook con Mistral. La submission individual la subo a Kaggle como referencia del mejor encoder antes de pasar al voting.